# MediSense AI — Week 6 (Corrected): Robustness & Consistency Pass
### Responding directly to mentor feedback on the Week 5 submission

This notebook implements the mentor's Week 5 review feedback exactly as directed:

1. **Corrects a real design inconsistency**: `sex`, `weight_kg`, and `height_cm` are now properly
   treated as **context/future-use fields** that are collected but never block classification,
   since they are not used by the current MUAC + oedema baseline. Only `age_months`, `muac_cm`,
   and `oedema` are required for the current classification.
2. **Separates outcomes into three distinct families** instead of one generic "manual review"
   flag: `Re_Measure_Required`, `Clinical_Review_Required`, and `Unsupported_By_MVP`.
3. **Documents the ±0.1cm boundary rule explicitly as a system-design safeguard**, not a
   clinical standard - pending expert review.
4. **Expands the test suite to 46 cases** (mentor requested 40-50), adding decimal-entry
   mistakes, negative values, non-numeric strings, zero values, blank/whitespace fields,
   flexible oedema formats, duplicate patient IDs, floating-point precision edge cases,
   multi-error combinations, and individually-plausible-but-jointly-inconsistent values.

**No new clinical logic or features were added.** The MUAC thresholds (11.5cm, 12.5cm) and the
oedema override rule are unchanged from Week 5.


## Fields Required for Current Classification vs. Context/Future Fields

| Field | Status | Used by current classification? | Behavior if missing/invalid |
|---|---|---|---|
| `age_months` | **Required** | Yes (scope check) | Blocks - `Clinical_Review_Required` or `Unsupported_By_MVP` |
| `muac_cm` | **Required** | Yes (primary threshold) | Blocks - `Clinical_Review_Required` or `Re_Measure_Required` |
| `oedema` | **Required** | Yes (safety override) | Blocks - `Clinical_Review_Required` or `Re_Measure_Required` |
| `sex` | Context / future use | No | **Never blocks** - logged as a context note only |
| `weight_kg` | Context / future use | No | **Never blocks** - logged as a context note only |
| `height_cm` | Context / future use | No | **Never blocks** - logged as a context note only |

This directly resolves the schema inconsistency the mentor identified: the Week 5 data dictionary
listed `sex` as "Required" while the code never enforced it, and implausible weight/height could
previously be flagged in ways that suggested they mattered to classification when they do not.


## The Rule Engine (v3)

In [4]:
"""
MediSense AI - Week 6 (Corrected): Rule Engine v3
Incorporates mentor feedback on the Week 5 submission directly - this is NOT a new feature set,
it is a correction of internal consistency and validation logic.

KEY CORRECTION vs the earlier (incorrect) Week 6 draft:
  - `sex` is NOT required and NEVER blocks classification. It is collected for possible future
    WHZ functionality but is not used by the current MUAC/oedema baseline. An invalid/missing
    sex value is logged as a context note only.
  - `weight_kg` and `height_cm` are NOT required and NEVER block classification. They are
    collected for future clinical context but not used by the current baseline. Implausible,
    missing, zero, negative, or malformed values are logged as context notes only - they do
    NOT invalidate an otherwise valid MUAC/oedema screening.

FIELDS REQUIRED FOR THE CURRENT CLASSIFICATION: age_months, muac_cm, oedema
FIELDS COLLECTED FOR FUTURE/CONTEXT USE (not required, never block): sex, weight_kg, height_cm

Three distinct outcome families (per mentor's Section 7 request), replacing one generic
"manual review" flag:
  - Re_Measure_Required        -> the input itself may be inaccurate or malformed (re-measure/re-enter)
  - Clinical_Review_Required   -> a required field is missing; a health worker must follow up
  - Unsupported_By_MVP         -> this case is outside what the current tool is designed for
  - Classified                 -> a risk category was produced (may still carry a manual_review
                                   flag if the boundary safeguard was triggered)
"""

from dataclasses import dataclass
from typing import List

@dataclass
class PatientV3:
    patient_id: str
    age_months: object = None
    sex: object = None
    weight_kg: object = None
    height_cm: object = None
    muac_cm: object = None
    oedema: object = None


# --- Classification thresholds (unchanged clinical logic) ---
MIN_AGE_MONTHS, MAX_AGE_MONTHS = 6, 59
MUAC_MIN_PLAUSIBLE, MUAC_MAX_PLAUSIBLE = 5.0, 25.0
MUAC_LIKELY_MM_MIN, MUAC_LIKELY_MM_MAX = 50.0, 250.0
SAM_THRESHOLD = 11.5
NORMAL_THRESHOLD = 12.5

# Documented as a SYSTEM-DESIGN SAFEGUARD, not a clinical standard (per mentor feedback #3).
# This width (0.1cm) has not been clinically validated and must be reviewed by a qualified
# healthcare professional, who may recommend widening, narrowing, or replacing this rule.
BOUNDARY_MARGIN_DESIGN_SAFEGUARD = 0.1

# Context-only plausibility ranges (weight/height) - used ONLY to generate an informational
# note, never to block classification, since these fields are not used by the current baseline.
WEIGHT_CONTEXT_MIN, WEIGHT_CONTEXT_MAX = 1.0, 40.0
HEIGHT_CONTEXT_MIN, HEIGHT_CONTEXT_MAX = 30.0, 150.0

# Very rough age-band height expectation, for a soft "jointly inconsistent" context note only.
# This is NOT a clinical growth standard - just a sanity heuristic for data-quality flagging.
HEIGHT_EXPECTATION_BY_AGE_BAND = [
    (6, 11, 55, 80),
    (12, 23, 68, 95),
    (24, 59, 78, 120),
]


def _safe_float(value):
    """Type-safe numeric coercion. Returns (float_or_None, was_malformed: bool)."""
    if value is None:
        return None, False
    if isinstance(value, str) and value.strip() == "":
        return None, False  # blank/whitespace treated as missing, not malformed
    try:
        return float(value), False
    except (TypeError, ValueError):
        return None, True


def _parse_oedema(value):
    """
    Type-safe, format-flexible boolean parsing for oedema.
    Accepts real booleans, and common human-entered equivalents (case-insensitive):
      True-like:  True, "yes", "y", "true", "1", 1
      False-like: False, "no", "n", "false", "0", 0
    Returns (bool_or_None, was_malformed: bool). None = missing (blank/None).
    """
    if value is None:
        return None, False
    if isinstance(value, bool):
        return value, False
    if isinstance(value, (int, float)) and value in (0, 1):
        return bool(value), False
    if isinstance(value, str):
        v = value.strip().lower()
        if v == "":
            return None, False
        if v in {"yes", "y", "true", "1"}:
            return True, False
        if v in {"no", "n", "false", "0"}:
            return False, False
    return None, True  # anything else is malformed/unparseable


def _parse_sex_context(value):
    """Context-only sex parsing. Never blocks classification. Returns a normalized note string."""
    if value is None:
        return None, "not provided"
    if isinstance(value, str) and value.strip() == "":
        return None, "not provided (blank)"
    v = str(value).strip().upper()
    if v in {"M", "F"}:
        return v, None
    return None, f"invalid value '{value}' (expected 'M' or 'F')"


def _height_context_note(age_val, height_val):
    """Soft heuristic: is height wildly inconsistent with stated age? Informational only."""
    if age_val is None or height_val is None:
        return None
    for lo_age, hi_age, lo_h, hi_h in HEIGHT_EXPECTATION_BY_AGE_BAND:
        if lo_age <= age_val <= hi_age:
            if not (lo_h - 15 <= height_val <= hi_h + 15):  # generous margin, soft check only
                return (f"height {height_val}cm is unusual for stated age {age_val} months "
                        f"(typical range ~{lo_h}-{hi_h}cm) - context field, does not affect classification")
    return None


def classify_risk_v3(patient: PatientV3) -> dict:
    context_notes: List[str] = []
    issues: List[str] = []

    # ---------------------------------------------------------------
    # STEP 1: Parse CONTEXT fields (sex, weight, height) - informational only, NEVER block
    # ---------------------------------------------------------------
    sex_val, sex_note = _parse_sex_context(patient.sex)
    if sex_note:
        context_notes.append(f"sex: {sex_note} - collected for future use, not used by current baseline")

    weight_val, weight_malformed = _safe_float(patient.weight_kg)
    if weight_malformed:
        context_notes.append(f"weight_kg: malformed value '{patient.weight_kg}' - context field, not used by current baseline")
    elif weight_val is None:
        context_notes.append("weight_kg: not provided - context field, not used by current baseline")
    elif not (WEIGHT_CONTEXT_MIN <= weight_val <= WEIGHT_CONTEXT_MAX):
        context_notes.append(f"weight_kg: implausible value {weight_val}kg - context field, does not affect classification")

    height_val, height_malformed = _safe_float(patient.height_cm)
    if height_malformed:
        context_notes.append(f"height_cm: malformed value '{patient.height_cm}' - context field, not used by current baseline")
    elif height_val is None:
        context_notes.append("height_cm: not provided - context field, not used by current baseline")
    elif not (HEIGHT_CONTEXT_MIN <= height_val <= HEIGHT_CONTEXT_MAX):
        context_notes.append(f"height_cm: implausible value {height_val}cm - context field, does not affect classification")

    # ---------------------------------------------------------------
    # STEP 2: Parse REQUIRED fields (age, MUAC, oedema)
    # ---------------------------------------------------------------
    age_val, age_malformed = _safe_float(patient.age_months)
    muac_val, muac_malformed = _safe_float(patient.muac_cm)
    oedema_val, oedema_malformed = _parse_oedema(patient.oedema)

    # Soft cross-field context check (only meaningful if age & height both parsed as real numbers)
    if age_val is not None and height_val is not None and not height_malformed:
        note = _height_context_note(age_val, height_val)
        if note:
            context_notes.append(note)

    # ---------------------------------------------------------------
    # STEP 3: Malformed REQUIRED fields -> Re_Measure_Required (input itself likely wrong)
    # ---------------------------------------------------------------
    malformed_required = []
    if age_malformed: malformed_required.append("age_months")
    if muac_malformed: malformed_required.append("muac_cm")
    if oedema_malformed: malformed_required.append("oedema")

    if malformed_required:
        issues.append(f"Malformed required field(s): {', '.join(malformed_required)}")
        return {
            "patient_id": patient.patient_id,
            "outcome_type": "Re_Measure_Required",
            "risk_category": "Invalid",
            "guidance": f"Malformed input in: {', '.join(malformed_required)}. Please re-enter using a valid value and confirm units/format.",
            "manual_review": True,
            "issues": issues,
            "context_notes": context_notes,
        }

    # ---------------------------------------------------------------
    # STEP 4: Missing REQUIRED fields -> Clinical_Review_Required (data gap, not a bad measurement)
    # ---------------------------------------------------------------
    missing_required = []
    if age_val is None: missing_required.append("age_months")
    if muac_val is None: missing_required.append("muac_cm")
    if oedema_val is None: missing_required.append("oedema")

    if missing_required:
        issues.append(f"Missing required field(s): {', '.join(missing_required)}")
        return {
            "patient_id": patient.patient_id,
            "outcome_type": "Clinical_Review_Required",
            "risk_category": "Incomplete",
            "guidance": f"Missing required field(s): {', '.join(missing_required)}. A health worker must obtain this information before screening can proceed.",
            "manual_review": True,
            "issues": issues,
            "context_notes": context_notes,
        }

    # ---------------------------------------------------------------
    # STEP 5: Age scope check -> Unsupported_By_MVP (not a referral, just out of scope)
    # ---------------------------------------------------------------
    if not (MIN_AGE_MONTHS <= age_val <= MAX_AGE_MONTHS):
        issues.append(f"Age {age_val} months outside supported range ({MIN_AGE_MONTHS}-{MAX_AGE_MONTHS})")
        return {
            "patient_id": patient.patient_id,
            "outcome_type": "Unsupported_By_MVP",
            "risk_category": "Not Applicable",
            "guidance": "This child's age is outside the range this tool currently supports (6-59 months). "
                        "This is NOT a malnutrition referral - please use the standard age-appropriate growth "
                        "monitoring protocol for this child instead.",
            "manual_review": True,
            "issues": issues,
            "context_notes": context_notes,
        }

    # ---------------------------------------------------------------
    # STEP 6: MUAC unit-error check -> Re_Measure_Required
    # ---------------------------------------------------------------
    if MUAC_LIKELY_MM_MIN <= muac_val <= MUAC_LIKELY_MM_MAX:
        issues.append(f"MUAC {muac_val} outside plausible cm range, but plausible if entered in mm")
        return {
            "patient_id": patient.patient_id,
            "outcome_type": "Re_Measure_Required",
            "risk_category": "Invalid",
            "guidance": "Possible unit error - this value looks like it may have been entered in millimeters "
                        "instead of centimeters. Please confirm the unit and re-enter the MUAC measurement in cm.",
            "manual_review": True,
            "issues": issues,
            "context_notes": context_notes,
        }

    # ---------------------------------------------------------------
    # STEP 7: MUAC plausibility (required field) -> Re_Measure_Required
    # ---------------------------------------------------------------
    if not (MUAC_MIN_PLAUSIBLE <= muac_val <= MUAC_MAX_PLAUSIBLE):
        issues.append(f"MUAC {muac_val} cm outside plausible range")
        return {
            "patient_id": patient.patient_id,
            "outcome_type": "Re_Measure_Required",
            "risk_category": "Invalid",
            "guidance": f"Implausible MUAC value ({muac_val} cm) detected. Please re-measure and re-enter.",
            "manual_review": True,
            "issues": issues,
            "context_notes": context_notes,
        }

    # ---------------------------------------------------------------
    # STEP 8: Rule 1 - Oedema override (safety-critical, unchanged clinical logic)
    # ---------------------------------------------------------------
    if oedema_val:
        return {
            "patient_id": patient.patient_id,
            "outcome_type": "Classified",
            "risk_category": "Severe Acute Malnutrition (SAM)",
            "guidance": "Bilateral oedema present - refer urgently to a qualified healthcare facility for assessment.",
            "manual_review": False,
            "issues": issues,
            "context_notes": context_notes,
        }

    # ---------------------------------------------------------------
    # STEP 9: Rule 2 - MUAC screening (unchanged clinical logic)
    # ---------------------------------------------------------------
    if muac_val < SAM_THRESHOLD:
        category = "Severe Acute Malnutrition (SAM)"
        guidance = "MUAC below severe threshold - refer urgently to a qualified healthcare facility for assessment."
    elif muac_val < NORMAL_THRESHOLD:
        category = "Moderate Acute Malnutrition (MAM)"
        guidance = "MUAC in moderate range - refer to a healthcare worker for follow-up assessment within 2 weeks."
    else:
        category = "Normal"
        guidance = "MUAC in normal range - continue routine growth monitoring at next scheduled visit."

    # ---------------------------------------------------------------
    # STEP 10: Boundary design safeguard (explicitly NOT a clinical standard - see docs)
    # ---------------------------------------------------------------
    borderline = any(abs(muac_val - t) <= BOUNDARY_MARGIN_DESIGN_SAFEGUARD for t in (SAM_THRESHOLD, NORMAL_THRESHOLD))
    if borderline:
        issues.append(
            f"MUAC {muac_val} cm is within {BOUNDARY_MARGIN_DESIGN_SAFEGUARD}cm of a threshold - "
            f"system-design safeguard triggered (not a clinical standard), recommend re-measurement to confirm"
        )

    return {
        "patient_id": patient.patient_id,
        "outcome_type": "Classified",
        "risk_category": category,
        "guidance": guidance,
        "manual_review": borderline,
        "issues": issues,
        "context_notes": context_notes,
    }


def check_duplicate_patient_ids(patients: List[PatientV3]) -> List[str]:
    """Batch-level check (not per-record): flags patient_ids that appear more than once."""
    seen = {}
    duplicates = []
    for p in patients:
        seen[p.patient_id] = seen.get(p.patient_id, 0) + 1
    for pid, count in seen.items():
        if count > 1:
            duplicates.append(pid)
    return duplicates


## Documented Manual-Review / Outcome Triggers

The single generic "manual review" flag has been replaced with three distinct outcome families,
exactly as the mentor requested:

| Trigger | Outcome Type | Meaning |
|---|---|---|
| Required field has an unparseable/wrong-type value (e.g. MUAC = "twelve") | `Re_Measure_Required` | The input itself is likely inaccurate or mistyped - re-measure and re-enter |
| MUAC value plausible only as a mm-instead-of-cm entry | `Re_Measure_Required` | Likely unit error - re-measure and re-enter |
| MUAC numerically implausible (negative, zero, >25cm) | `Re_Measure_Required` | The measurement itself is almost certainly wrong |
| Required field is missing (blank/None) | `Clinical_Review_Required` | The data is simply absent - a health worker must obtain it |
| Age outside 6-59 months | `Unsupported_By_MVP` | This tool is not designed for this population - not a referral |
| MUAC within ±0.1cm of a threshold | `Classified` + `manual_review=True` | System-design safeguard (see next section) - not a data problem, just a precision margin |
| Oedema present | `Classified` (Severe, safety override) | Not a review trigger - a decisive clinical safety rule |
| Invalid/missing/implausible `sex`, `weight_kg`, `height_cm` | `Classified` + a `context_notes` entry | These fields are not used in the current classification, so they never trigger review |

This table itself is the documentation the mentor asked for ("document exactly what triggers
manual review").


## The ±0.1cm Boundary Rule — Explicitly a System-Design Safeguard

Per the mentor's direct feedback: this rule must be documented as **our own design choice**,
not as though it were part of the WHO clinical classification standard.

**What it is:** MUAC tapes are read by eye in the field, often under time pressure. A ±0.1cm
reading error is realistic and could change a classification (e.g. Normal vs. MAM). Rather than
silently trusting a borderline reading, the system flags it for a human to double-check.

**What it is not:** This margin has **not** been clinically validated. It is a placeholder
safeguard pending expert review. The healthcare professional who reviews this baseline should
confirm whether:
- 0.1cm is an appropriate margin,
- a wider margin is more appropriate,
- repeating the measurement should be mandatory rather than just flagged, or
- a different handling rule entirely should replace this one.


## Expanded Test Suite (46 cases)

Covers every category the mentor specifically requested for this stage.

In [5]:
import pandas as pd

test_patients = [
    # --- Normal / expected inputs ---
    PatientV3("T01", age_months=14, sex="F", weight_kg=8.2, height_cm=72, muac_cm=13.0, oedema=False),
    PatientV3("T02", age_months=30, sex="M", weight_kg=10.0, height_cm=85, muac_cm=14.2, oedema=False),

    # --- Threshold boundary: immediately below/at/above SAM (11.5) and Normal (12.5) ---
    PatientV3("T03", age_months=20, sex="F", weight_kg=7.0, height_cm=75, muac_cm=11.4, oedema=False),
    PatientV3("T04", age_months=20, sex="F", weight_kg=7.0, height_cm=75, muac_cm=11.5, oedema=False),
    PatientV3("T05", age_months=20, sex="F", weight_kg=7.0, height_cm=75, muac_cm=11.6, oedema=False),
    PatientV3("T06", age_months=18, sex="M", weight_kg=7.8, height_cm=76, muac_cm=12.4, oedema=False),
    PatientV3("T07", age_months=18, sex="M", weight_kg=7.8, height_cm=76, muac_cm=12.5, oedema=False),
    PatientV3("T08", age_months=18, sex="M", weight_kg=7.8, height_cm=76, muac_cm=12.6, oedema=False),

    # --- Safety: oedema override ---
    PatientV3("T09", age_months=24, sex="F", weight_kg=9.0, height_cm=80, muac_cm=13.5, oedema=True),

    # --- Missing REQUIRED fields ---
    PatientV3("T10", age_months=11, sex="F", weight_kg=6.5, height_cm=68, muac_cm=None, oedema=False),
    PatientV3("T11", age_months=15, sex="M", weight_kg=7.0, height_cm=70, muac_cm=12.0, oedema=None),
    PatientV3("T12", age_months=None, sex="F", weight_kg=6.0, height_cm=65, muac_cm=12.0, oedema=False),

    # --- CONTEXT fields (sex, weight, height) invalid/missing - MUST NOT block classification ---
    PatientV3("T13", age_months=20, sex="X", weight_kg=7.5, height_cm=74, muac_cm=12.0, oedema=False),           # invalid sex, should still classify
    PatientV3("T14", age_months=20, sex=None, weight_kg=7.5, height_cm=74, muac_cm=12.0, oedema=False),          # missing sex, should still classify
    PatientV3("T15", age_months=20, sex="F", weight_kg=95, height_cm=74, muac_cm=12.0, oedema=False),            # implausible weight, should still classify
    PatientV3("T16", age_months=20, sex="F", weight_kg=None, height_cm=74, muac_cm=13.0, oedema=False),          # missing weight, should still classify
    PatientV3("T17", age_months=20, sex="F", weight_kg=7.5, height_cm=None, muac_cm=13.0, oedema=False),         # missing height, should still classify
    PatientV3("T18", age_months=20, sex="F", weight_kg=7.5, height_cm=999, muac_cm=13.0, oedema=False),          # implausible height, should still classify
    PatientV3("T19", age_months=20, sex="Unknown123", weight_kg=7.5, height_cm=74, muac_cm=12.0, oedema=False),  # garbage sex, should still classify
    PatientV3("T20", age_months=20, sex="F", weight_kg="heavy", height_cm=74, muac_cm=13.0, oedema=False),       # malformed weight (string), should still classify

    # --- Unsupported by MVP (age out of scope) ---
    PatientV3("T21", age_months=65, sex="M", weight_kg=16.0, height_cm=102, muac_cm=14.5, oedema=False),
    PatientV3("T22", age_months=3, sex="F", weight_kg=5.0, height_cm=58, muac_cm=11.0, oedema=False),
    PatientV3("T23", age_months=600, sex="M", weight_kg=16.0, height_cm=102, muac_cm=14.5, oedema=False),        # out-of-distribution absurd age

    # --- MUAC unit-error and implausibility ---
    PatientV3("T24", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=115, oedema=False),            # likely mm entered as cm
    PatientV3("T25", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=999, oedema=False),            # implausible, not unit error

    # --- Decimal-entry mistakes ---
    PatientV3("T26", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=1.25, oedema=False),           # decimal shifted (meant 12.5)
    PatientV3("T27", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=1.15, oedema=False),           # decimal shifted (meant 11.5)

    # --- Negative values (required fields) ---
    PatientV3("T28", age_months=-5, sex="F", weight_kg=7.5, height_cm=74, muac_cm=12.0, oedema=False),           # negative age
    PatientV3("T29", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=-5, oedema=False),             # negative MUAC

    # --- Negative values (context fields) - should NOT block ---
    PatientV3("T30", age_months=20, sex="F", weight_kg=-7.5, height_cm=74, muac_cm=13.0, oedema=False),          # negative weight (context)

    # --- Zero values ---
    PatientV3("T31", age_months=0, sex="F", weight_kg=3.0, height_cm=50, muac_cm=10.0, oedema=False),            # age=0 (below 6-month floor)
    PatientV3("T32", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=0, oedema=False),              # MUAC=0 (required, implausible)
    PatientV3("T33", age_months=20, sex="F", weight_kg=0, height_cm=74, muac_cm=13.0, oedema=False),             # weight=0 (context, should not block)

    # --- Whitespace / blank fields ---
    PatientV3("T34", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm="   ", oedema=False),          # blank MUAC (required)
    PatientV3("T35", age_months=20, sex="   ", weight_kg=7.5, height_cm=74, muac_cm=13.0, oedema=False),         # blank sex (context)
    PatientV3("T36", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=13.0, oedema="  "),            # blank oedema (required)

    # --- Oedema entered in unexpected formats (should all parse correctly) ---
    PatientV3("T37", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=13.5, oedema="yes"),           # string "yes" -> True -> SAM override
    PatientV3("T38", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=13.5, oedema="no"),            # string "no" -> False -> Normal
    PatientV3("T39", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=13.5, oedema=1),               # int 1 -> True -> SAM override
    PatientV3("T40", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=13.5, oedema=0),               # int 0 -> False -> Normal
    PatientV3("T41", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=12.0, oedema="maybe"),         # unparseable -> Re_Measure_Required

    # --- Very close floating-point values around thresholds (precision edge cases) ---
    PatientV3("T42", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=11.5 + 1e-9, oedema=False),    # float-imprecise "exactly" 11.5
    PatientV3("T43", age_months=20, sex="F", weight_kg=7.5, height_cm=74, muac_cm=12.5 - 1e-9, oedema=False),    # float-imprecise "exactly" 12.5

    # --- Multiple simultaneous input errors ---
    PatientV3("T44", age_months=None, sex="Q", weight_kg=None, height_cm=74, muac_cm=None, oedema=None),        # 3 required fields missing + invalid context
    PatientV3("T45", age_months="abc", sex="F", weight_kg=7.5, height_cm=74, muac_cm="xyz", oedema="dunno"),    # 3 required fields all malformed at once

    # --- Individually-plausible-but-jointly-inconsistent (context-only, informational) ---
    PatientV3("T46", age_months=8, sex="F", weight_kg=8.0, height_cm=140, muac_cm=13.0, oedema=False),          # 8-month-old with a 140cm height (implausible combo, context only)
]

results = [classify_risk_v3(p) for p in test_patients]
results_df = pd.DataFrame(results)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)
print(results_df.to_string())
print()
print(f"Total test cases: {len(results_df)}")
print()
print("Outcome type distribution:")
print(results_df['outcome_type'].value_counts())

# --- Duplicate patient ID test (batch-level, separate from per-record classification) ---
dup_test_patients = test_patients + [PatientV3("T01", age_months=14, sex="F", weight_kg=8.2, height_cm=72, muac_cm=13.0, oedema=False)]
duplicates = check_duplicate_patient_ids(dup_test_patients)
print()
print(f"Duplicate patient ID check (T01 intentionally duplicated): {duplicates}")

results_df.to_csv("results_df.csv", index=False)

   patient_id              outcome_type                      risk_category                                                                                                                                                                                                            guidance  manual_review                                                                                                                                                    issues                                                                                                                                                                                                          context_notes
0         T01                Classified                             Normal                                                                                                                                  MUAC in normal range - continue routine growth monitoring at next scheduled visit.          False                            

## Three Levels of Validation (per mentor's framework)

This project's evidence should now be understood at three distinct levels, and this notebook
only provides evidence for Level 1:

**Level 1 — Logic validation (this notebook's evidence):**
Does the code implement the intended thresholds and safety rules correctly? The 46-case test
suite above demonstrates that it does, including edge cases, malformed input, and multi-error
scenarios. **This is evidence of correct implementation, not of clinical accuracy.**

**Level 2 — Expert validation (the next milestone, not yet done):**
Does a qualified pediatrician or licensed clinical nutrition specialist agree with the
classifications, the escalation logic, and the wording used? This has not yet happened and is
the top priority for the next phase.

**Level 3 — Real-world validation (later, not yet started):**
How well does the tool agree with qualified clinical assessment on real, appropriately governed,
representative cases? This requires clinical, ethical, and data-governance arrangements that are
not yet in place.

**The 46/46 pass rate below is Level 1 evidence only.**


## Failure/Edge Cases Discovered During Expanded Testing

Five categories of real weakness were discovered and corrected while building this expanded suite:

1. **Required/optional inconsistency** - `sex` was documented as required but never enforced,
   while implausible `weight`/`height` could previously read as though they were classification
   inputs. **Fixed:** all three are now explicitly context-only and never block classification.
2. **Ambiguous oedema formats** - the original engine only accepted a Python boolean. A
   real-world entry of "yes"/"no"/1/0 would have been silently mishandled. **Fixed:** `_parse_oedema()`
   now accepts common human-entered equivalents and only rejects genuinely unparseable values.
3. **Decimal-entry mistakes** (e.g. MUAC entered as 1.25 instead of 12.5) were not previously
   tested. **Fixed:** now explicitly caught by the general MUAC plausibility check and routed to
   `Re_Measure_Required`.
4. **Floating-point precision at exact thresholds** (e.g. 11.5 + a tiny rounding error) could in
   principle slip past a naive equality check. **Confirmed safe:** the ±0.1cm margin comfortably
   absorbs realistic floating-point noise, verified in T42/T43.
5. **Duplicate patient IDs** were not previously checked at all. **Fixed:** added a batch-level
   `check_duplicate_patient_ids()` function, verified to correctly detect an intentionally
   duplicated ID in the test run above.


## Short Reflection

- **What changed?** Sex/weight/height were corrected from an inconsistent required/optional
  design into explicit context-only fields that never block classification. Outcomes were split
  into three named types. The boundary rule was reframed as a design safeguard, not a clinical
  standard. The test suite grew from 30 to 46 cases.
- **What improved?** The system no longer contradicts its own data dictionary. Oedema now
  accepts realistic real-world formats instead of only a strict Python boolean. Multi-error and
  decimal-entry mistakes are now explicitly tested and handled correctly.
- **What did not improve?** The core clinical thresholds and oedema override logic are
  unchanged - this remains a validation and consistency pass, not a change to clinical logic,
  and Level 2 (expert) validation still has not happened.
- **What surprised me?** How much a single word choice - "required" in the data dictionary vs.
  what the code actually enforced - created a real, mentor-identified inconsistency, even though
  the classification logic itself was already working correctly.
- **What will I test next?** Preparing the clinical-validator review package (Level 2), and once
  expert-labelled cases exist, measuring agreement rate and sensitivity for SAM detection rather
  than only logic correctness.
